# 01 · Explorer avant de décider : que contient vraiment le fichier ?

**Séance 3 · deux blocs de 20 minutes en séance, 40 minutes en autonomie.**

Question directrice du cours : **« Qui sont nos clients, et à qui devrions-nous envoyer la prochaine campagne ? »** Avant de segmenter ou de prédire quoi que ce soit, il faut savoir ce que vaut le fichier. C'est l'**analyse exploratoire** (EDA) : on regarde tout, on note tout, et **on ne corrige rien dans ce notebook**. Les décisions de nettoyage viendront après, une par une, en séance 4.

Rappel : le fichier est le même qu'au notebook 00 (2 240 clients, 29 colonnes, **données simulées**).

**Bloc 1 — Le fichier tel qu'il est** : types, trous, doublons, forme d'une variable.
**Bloc 2 — Chasse aux anomalies et premiers liens avec la cible** : ce qui cloche, qui répond, ce que le fichier ne dit pas.

## Bloc 1 — Le fichier tel qu'il est

### 1. Charger

Même première cellule qu'au notebook 00. On y ajoute `matplotlib`, la bibliothèque qui dessine les graphiques.

In [ ]:
# Première cellule de chaque notebook : récupérer le module du cours, puis charger le fichier brut.
import os
import urllib.request

URL_MODULE = "https://raw.githubusercontent.com/2501Pr0ject/Data4Sup2Pub/main/notebooks/preparation.py"
if not os.path.exists("preparation.py"):
    urllib.request.urlretrieve(URL_MODULE, "preparation.py")

import pandas as pd                   # manipulation de tableaux
import matplotlib.pyplot as plt       # graphiques
import preparation as prep

brut = prep.charger()                 # le tableau brut : rien n'est modifié dans ce notebook

### 2. De quel type sont les variables ?

**Question marketing :** *quelles informations sont des nombres sur lesquels on peut calculer, lesquelles sont des mots, lesquelles des dates ?*

**La notion.** Chaque colonne a un **type**. Les nombres (`int64`, `float64`) se prêtent aux moyennes et aux graphiques ; les textes (`object`) se comptent par modalité ; une **date rangée comme du texte** ne se calcule pas tant qu'on ne l'a pas convertie. Le type que pandas devine n'est pas toujours celui qu'on voudrait.

In [ ]:
# Combien de colonnes de chaque type, puis lesquelles sont du texte
print(brut.dtypes.value_counts().to_string(), "\n")
print("Colonnes de type texte :", list(brut.dtypes[brut.dtypes == "object"].index))
print("\nExemples de Dt_Customer :", list(brut["Dt_Customer"].head(3)))

**Question 1.** `Dt_Customer` est rangée comme du texte : que ne peut-on pas faire avec, en l'état ? À l'inverse, citez deux colonnes rangées comme des nombres qui sont en réalité des catégories (oui / non, ou un petit nombre de valeurs).

### 3. Qu'est-ce qui manque ?

**Question marketing :** *peut-on faire confiance à toutes les cases ?*

**La notion.** Une **valeur manquante** est une case vide (`NaN`). On commence par les compter. Puis on se demande si les clients concernés ressemblent aux autres, en gardant en tête qu'avec **peu de cas**, un écart peut n'être que du hasard : la question « combien ? » précède toujours la question « différents ? ».

In [ ]:
# Nombre de cases vides par colonne (on n'affiche que les colonnes concernées)
manquants = brut.isna().sum()
print(manquants[manquants > 0].to_string(), "\n")

# Les clients sans revenu ressemblent-ils aux autres ?
sans_revenu = brut["Income"].isna()
comparaison = pd.DataFrame({
    "revenu manquant": [sans_revenu.sum(), brut.loc[sans_revenu, "Response"].mean(), brut.loc[sans_revenu, "Recency"].mean()],
    "revenu renseigné": [(~sans_revenu).sum(), brut.loc[~sans_revenu, "Response"].mean(), brut.loc[~sans_revenu, "Recency"].mean()],
}, index=["nombre de clients", "taux de réponse", "récence moyenne (jours)"])
comparaison.round(3)

Un répondant sur 24, contre 15 % en général : est-ce un écart réel ou un effet du petit nombre ? Pour le savoir, on tire **10 000 fois 24 clients au hasard** dans le fichier et on regarde combien de ces tirages font « au moins aussi bizarre ».

In [ ]:
import numpy as np

rng = np.random.default_rng(42)                      # graine fixe : tout le monde obtient les mêmes tirages
reponses = brut["Response"].to_numpy()
recences = brut["Recency"].to_numpy()
tirages = np.array([rng.choice(len(brut), size=24, replace=False) for _ in range(10_000)])

nb_repondants = reponses[tirages].sum(axis=1)        # répondants dans chaque tirage de 24 clients
recence_moyenne = recences[tirages].mean(axis=1)     # récence moyenne de chaque tirage

print(f"Répondants attendus parmi 24 clients pris au hasard : {24 * reponses.mean():.1f}")
print(f"Tirages avec au plus 1 répondant                   : {(nb_repondants <= 1).mean():.1%}")
print(f"Tirages avec une récence moyenne d'au moins 58 jours : {(recence_moyenne >= 58).mean():.1%}")

**Question 2.** Combien de clients n'ont pas de revenu ? Peut-on affirmer qu'ils sont différents des autres ? Trois options sont possibles pour ces lignes : les supprimer, remplacer le revenu par la médiane, ou garder la case vide. Quelle est la conséquence de chacune ?

### 4. Des doublons invisibles

**Question marketing :** *chaque ligne est-elle bien un client différent ?*

**La notion.** `duplicated()` repère les lignes **entièrement** identiques. Mais si deux lignes décrivent le même client avec deux identifiants différents, elles ne sont pas « entièrement » identiques : il faut comparer **toutes les colonnes sauf l'identifiant**.

In [ ]:
print("Lignes entièrement identiques :", brut.duplicated().sum())
print("Identifiants distincts :", brut["ID"].nunique(), "sur", len(brut), "lignes")

# Même comparaison en ignorant l'identifiant
colonnes_sans_id = [c for c in brut.columns if c != "ID"]
jumeaux = brut[brut.duplicated(subset=colonnes_sans_id, keep=False)]
print("Lignes qui ont un jumeau parfait (hors ID) :", len(jumeaux))
print("Lignes en trop si l'on ne garde qu'une copie de chaque profil :", brut.duplicated(subset=colonnes_sans_id).sum())

# Deux paires, pour voir
jumeaux.sort_values(colonnes_sans_id)[["ID", "Year_Birth", "Education", "Marital_Status", "Income",
                                        "Dt_Customer", "Recency", "MntWines", "Response"]].head(4)

**Question 3.** Pourquoi la vérification standard (`duplicated()`) n'a-t-elle rien trouvé ? Les deux lignes de chaque paire peuvent-elles être deux clients différents ? Imaginez qu'en séance 4 on entraîne un modèle sur une partie des clients et qu'on le teste sur l'autre : que se passe-t-il si un jumeau est de chaque côté ?

### 5. À quoi ressemble une variable ?

**Question marketing :** *combien dépense un client « typique » en vin ?*

**La notion.** Un **histogramme** montre comment les clients se répartissent : beaucoup de petits montants et quelques très gros, ou une masse au centre ? La **moyenne** additionne tout et divise ; la **médiane** coupe la population en deux. Quand la distribution est **asymétrique** (une longue queue à droite), la moyenne est tirée vers le haut et ne décrit plus le client typique.

**Formulaire.** La cellule ci-dessous a un menu déroulant (dans Colab, à droite du code). Choisissez une variable, relancez la cellule, observez. Commencez par `MntWines`, puis essayez `Income`.

In [ ]:
#@title Choisir la variable à regarder
variable = "MntWines"  #@param ["MntWines", "MntMeatProducts", "MntGoldProds", "Depenses_total", "Income", "Recency", "Year_Birth"]

colonnes_mnt = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
Depenses_total = brut[colonnes_mnt].sum(axis=1)      # somme des six rayons (même calcul que fera le module de préparation)
serie = Depenses_total if variable == "Depenses_total" else brut[variable].dropna()
moyenne, mediane = serie.mean(), serie.median()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(serie, bins=40, color="#3b6ea5", edgecolor="white", linewidth=0.5)
ax.axvline(moyenne, color="#c0392b", linewidth=2, label=f"moyenne = {moyenne:,.0f}")
ax.axvline(mediane, color="#1f1f1f", linewidth=2, linestyle="--", label=f"médiane = {mediane:,.0f}")
ax.set_title(f"{variable} : comment se répartissent les {len(serie)} clients ?", loc="left")
ax.set_ylabel("nombre de clients")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

print(f"{(serie < moyenne).mean():.0%} des clients sont en dessous de la moyenne")

**Question 4.** Pour `MntWines` : quel chiffre donneriez-vous à la directrice marketing qui demande « combien dépense un client en vin ? », et pourquoi ? Passez ensuite le menu sur `Income` : la moyenne et la médiane sont presque égales alors que le fichier contient un revenu de 666 666. Comment est-ce possible ? La cellule suivante donne un indice.

In [ ]:
# Contre-exemple : le poids d'une seule valeur aberrante sur la moyenne d'un grand échantillon
revenu = brut["Income"].dropna()
print(f"Moyenne du revenu avec la valeur 666 666 : {revenu.mean():,.0f}")
print(f"Moyenne du revenu sans elle            : {revenu[revenu != 666666].mean():,.0f}")
print(f"Médiane (inchangée)                    : {revenu.median():,.0f}")

La même valeur, isolée, pèse-t-elle aussi peu sur le **lien** entre revenu et dépenses ? On mesure ce lien par la corrélation (0 : aucun lien ; 1 : lien parfait).

In [ ]:
avec = brut["Income"].notna()
sans = avec & (brut["Income"] != 666666)
print("Force du lien revenu–dépenses (corrélation, de 0 = aucun lien à 1 = lien parfait)")
print(f"   avec la valeur 666 666 : {brut.loc[avec, 'Income'].corr(Depenses_total[avec]):.2f}")
print(f"   sans elle              : {brut.loc[sans, 'Income'].corr(Depenses_total[sans]):.2f}")

**À retenir.** Une valeur aberrante isolée pèse **peu sur la moyenne** d'un grand échantillon (277 de décalage sur 2 216 clients), mais elle peut peser **lourd sur une relation** entre deux variables : un seul client au revenu de 666 666 qui ne dépense presque rien suffit à faire passer le lien revenu–dépenses de 0,79 à 0,67. C'est pour les relations, pas pour les moyennes, qu'une telle valeur est dangereuse.

## Bloc 2 — Chasse aux anomalies et premiers liens avec la cible

### 6. Chasse aux anomalies (guidée)

**Question marketing :** *que faut-il vérifier avant de croire ce fichier ?*

**La notion.** Une **anomalie** est une valeur qui ne devrait pas exister (une année de naissance 1893), qui ne veut rien dire (une situation familiale « YOLO ») ou qui n'apporte rien (une colonne identique pour tout le monde). Une anomalie n'est pas forcément une erreur : avant de décider, on compte combien de clients sont concernés. Quatre décisions possibles : **garder**, **supprimer**, **recoder**, **signaler sans toucher**.

In [ ]:
print("Année de naissance : min", brut["Year_Birth"].min(), "| max", brut["Year_Birth"].max())
print("Les 3 plus anciennes :", sorted(brut["Year_Birth"])[:3], "→ âge en 2014 :", [2014 - a for a in sorted(brut["Year_Birth"])[:3]])

print("\nRevenu : les 5 plus élevés :", [int(v) for v in sorted(brut["Income"].dropna(), reverse=True)[:5]])

print("\nSituation familiale :")
print(brut["Marital_Status"].value_counts().to_string())

print("\nValeurs distinctes de Z_CostContact :", brut["Z_CostContact"].nunique(),
      "| de Z_Revenue :", brut["Z_Revenue"].nunique(),
      "→ elles valent", brut["Z_CostContact"].iloc[0], "et", brut["Z_Revenue"].iloc[0], "pour tout le monde")

achats = brut["NumWebPurchases"] + brut["NumCatalogPurchases"] + brut["NumStorePurchases"]
print("\nAchats par canal : max web", brut["NumWebPurchases"].max(), "| max catalogue", brut["NumCatalogPurchases"].max(),
      "| clients sans aucun achat :", (achats == 0).sum(),
      "| dont avec des dépenses > 0 :", ((achats == 0) & (Depenses_total > 0)).sum())

**Question 5.** Complétez le tableau (une ligne par anomalie). Pour la décision, choisissez parmi garder / supprimer / recoder / signaler, et justifiez en une phrase.

| Anomalie | Combien de clients ? | Décision proposée | Pourquoi |
|---|---|---|---|
| Années de naissance 1893, 1899, 1900 | | | |
| Revenu de 666 666 | | | |
| Revenus entre 150 000 et 163 000 | | | |
| `Marital_Status` = Absurd, YOLO | | | |
| `Marital_Status` = Alone | | | |
| Colonnes `Z_CostContact`, `Z_Revenue` | | | |
| Clients à 0 achat mais dépenses > 0 | | | |
| 24 revenus manquants (section 3) | | | |
| 182 lignes jumelles (section 4) | | | |

### 7. Premiers liens avec la cible

**Question marketing :** *qui a répondu à la dernière campagne, et qu'est-ce qui les distingue ?*

**La notion.** Le **taux de réponse par groupe** est la moyenne de `Response` dans chaque groupe. Comparer ces taux montre des **associations** : un groupe répond plus qu'un autre. Cela ne dit pas pourquoi, ni ce qui se passerait si l'on changeait quelque chose.

In [ ]:
def taux_par(groupe, nom):
    # Nombre de clients et taux de réponse pour chaque valeur de `groupe`
    t = brut.groupby(groupe)["Response"].agg(clients="size", taux_reponse="mean")
    t.index.name = nom
    return t.round(3)

display(taux_par("Education", "Education"))

campagnes_acceptees = brut[["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]].sum(axis=1)
display(taux_par(campagnes_acceptees, "campagnes passées acceptées"))

# 10 répondants sur 11 : le hasard peut-il produire cela ? Même méthode qu'à la section 3.
tirages_11 = np.array([rng.choice(len(brut), size=11, replace=False) for _ in range(10_000)])
print("Tirages de 11 clients au hasard avec au moins 10 répondants :", int((reponses[tirages_11].sum(axis=1) >= 10).sum()), "sur 10 000")

**Question 6.** Quelle est la différence la plus forte ? Un client PhD répond davantage : est-ce son diplôme qui le fait répondre ? Les clients qui ont accepté 4 campagnes répondent à 91 % : que vaut ce chiffre, et que dit-il pour la question « à qui envoyer la prochaine campagne ? »

### 8. L'ancienneté : des médianes qui montent, un pouvoir explicatif faible

**Question marketing :** *les clients anciens dépensent-ils plus que les nouveaux ?*

**La notion.** Comparer des **médianes par tranche** répond à « un client ancien typique dépense-t-il plus ? ». Mais pour savoir si l'ancienneté **renseigne** sur les dépenses d'un client donné, il faut regarder la **dispersion** à l'intérieur de chaque tranche. Le graphique en boîtes le montre : chaque boîte contient la moitié centrale des clients de la tranche ; si les boîtes se chevauchent largement, connaître l'ancienneté d'un client ne dit presque rien de sa dépense.

L'ancienneté se calcule à partir de `Dt_Customer` ; faute de date d'extraction documentée, on prend comme référence la **dernière inscription observée**.

In [ ]:
date_inscription = pd.to_datetime(brut["Dt_Customer"], format="%d-%m-%Y")   # on dit à pandas : jour-mois-année
reference = date_inscription.max()
print("Inscriptions du", date_inscription.min().date(), "au", reference.date(), "→ date de référence :", reference.date())

Anciennete_mois = (reference.year - date_inscription.dt.year) * 12 + (reference.month - date_inscription.dt.month)
tranche = pd.cut(Anciennete_mois, bins=[-1, 5, 11, 17, 23], labels=["0-5 mois", "6-11 mois", "12-17 mois", "18-23 mois"])
tranche.name = "ancienneté"

par_tranche = Depenses_total.groupby(tranche, observed=True).agg(
    clients="size", quart_bas=lambda s: s.quantile(.25), mediane="median", quart_haut=lambda s: s.quantile(.75)).round(0)
par_tranche["taux_reponse"] = brut["Response"].groupby(tranche, observed=True).mean().round(3)
display(par_tranche)

fig, ax = plt.subplots(figsize=(8, 3.5))
groupes = [Depenses_total[tranche == t] for t in par_tranche.index]
ax.boxplot(groupes, showfliers=False, medianprops={"color": "#c0392b", "linewidth": 2})
ax.set_xticks(range(1, len(groupes) + 1))
ax.set_xticklabels(par_tranche.index)
ax.set_ylabel("dépenses totales")
ax.set_title("La médiane monte avec l'ancienneté, mais regardez la hauteur des boîtes", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

# Les tranches se recouvrent-elles ? Une phrase simple suffit.
mediane_anciens = Depenses_total[tranche == "18-23 mois"].median()
part_recents_au_dessus = (Depenses_total[tranche == "0-5 mois"] > mediane_anciens).mean()
print(f"{part_recents_au_dessus:.0%} des clients récents (0-5 mois) dépensent plus que la moitié des clients anciens (médiane {mediane_anciens:.0f}).")

**Question 7.** La médiane des dépenses passe de 158 (clients de moins de 6 mois) à 578 (18 mois et plus), et le taux de réponse de 9 % à 27 %. Pourtant un tiers des clients récents dépensent plus que la moitié des anciens, et les boîtes se chevauchent presque entièrement. Comment ces deux faits tiennent-ils ensemble ? Que répondez-vous à un collègue qui écrit « les clients anciens dépensent 3,5 fois plus » ?

### 9. Ce que le fichier ne dit pas : la chronologie

Pas de code ici, seulement une question qui comptera beaucoup en séance 4.

Les dépenses (`Mnt…`) portent sur « les 2 dernières années », `Recency` compte les jours depuis le dernier achat, `NumWebVisitsMonth` les visites « du mois dernier ». Deux dernières années **avant quoi** ? Le fichier ne donne **ni la date d'extraction, ni la date de la campagne** mesurée par `Response`. On sait seulement que la dernière inscription date du 29 juin 2014.

**Question 8.** Pour prédire qui répondra à la *prochaine* campagne, il faut des informations connues **avant** l'envoi. Pour chacune de ces variables, dites si elle était connue avant la campagne : certainement, probablement, ou on ne peut pas le savoir.

| Variable | Connue avant la campagne ? | Pourquoi |
|---|---|---|
| `Year_Birth`, `Education`, `Income` | | |
| `AcceptedCmp1` … `AcceptedCmp5` | | |
| `MntWines` et les autres dépenses | | |
| `Recency` | | |

### 10. Synthèse métier

- Le fichier est **utilisable mais pas tel quel** : 24 revenus manquants, 182 lignes jumelles, une poignée de valeurs impossibles, deux colonnes constantes. Chaque cas appelle une décision argumentée, pas un réflexe.
- Les dépenses sont **très asymétriques** : parler du client « typique », c'est donner la médiane. La moyenne sert au chiffre d'affaires.
- Le meilleur indice de réponse à une campagne, c'est la **réponse aux campagnes passées** (8 % → 91 %). Le diplôme, le revenu, l'ancienneté jouent moins, et aucun n'est une cause démontrée.
- Le fichier **ne date rien** : on ne sait pas si les comportements ont été mesurés avant ou après la campagne.

### 11. Limites

- Rien n'a été corrigé ici ; aucun chiffre de ce notebook n'est « propre ».
- Toutes les associations observées sont des **corrélations** sur une campagne passée, envoyée à 2 240 clients tirés au hasard ; elles ne disent pas ce qu'une campagne ciblée produirait.
- Les données sont simulées : certaines régularités (jumeaux parfaits, `Recency` répartie uniformément de 0 à 99) sont des traces de la fabrication du fichier, pas des faits de clientèle.

### En autonomie (40 minutes)

1. **Terminer la chasse aux anomalies** : le tableau de la question 5, complet, une ligne par anomalie, avec décision et justification. (15 min)
2. **Trois hypothèses** sur les clients, écrites sous la forme « *les clients qui [groupe, défini par une variable d'entrée] ont [une mesure : taux de réponse ou dépenses] plus élevé(e)* ». Exemple : « les clients avec des enfants dépensent moins en vin ». Pour chacune, dites quelle cellule de ce notebook permettrait de la vérifier, et ce que vous attendez. (15 min)
3. **Vérifiez-en une** avec la fonction `taux_par` (copiez la cellule de la section 7 et changez la colonne). Notez le résultat et ce qu'il ne prouve pas. (10 min)

Apportez vos trois hypothèses en séance 4 : elles serviront à choisir les variables du modèle.